# Week 1 · Environment & Tools：Python、Git、NumPy、seedの習慣

# Requirements: pip install numpy
# （Python 3.10+推奨。Gitはsystem toolであり、pip packageではありません。）

このnotebookは **environment self-check** です。Python、Git、core library、`zoro` package、そして何より後の全週が依存するdeterministic seedの習慣を確認します。最後に、checkごとに1点、合計0〜6の **readiness score** を表示します。先へ進む前に6にしてください。

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1])) # repo root
# Robust fallback: walk up to the real repo root if the kernel's cwd differs.
p = pathlib.Path.cwd()
while not (p / "zoro").is_dir() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

# Check 1, Python 3.10+.
import platform
print("Python:", sys.version.split()[0], "| platform:", platform.platform())
CHECK_PYTHON = 1 if sys.version_info >= (3, 10) else 0
print("OK: Python 3.10+" if CHECK_PYTHON else "FAIL: need Python 3.10+")

# Check 2, numpy and the zoro package import cleanly.
CHECK_LIBS = 0
try:
    import numpy as np
    import zoro
    from zoro import data
    print("numpy:", np.__version__, "| zoro:", zoro.__version__)
    CHECK_LIBS = 1
except Exception as e:
    print("FAIL: could not import numpy/zoro:", e)

### なぜ最初にenvironmentを整えるのか

Andrew Ngのskills mapでは **software engineering fundamentals** が2番目に置かれています。Zorostがこのskillをさらに率直に表現すると、coding agentがcodeを書く90秒の間に100個ものtradeoffが決まり、あなたはそれをdiffとしてreviewしなければなりません。実行できないものはreviewできません。reproducibleなenvironment、pinned interpreter、pinned dependencies、cleanなGit checkoutが、このprogram全体を支える床です。上のcheckはその床を *測定可能* にします。数字が6でなければ、何かが壊れています。

In [ ]:
# Check 3, zoro.data exposes the seven generators the whole program reuses.
CHECK_ZORO = 0
expected = ["carriers", "lanes", "shipments", "support_tickets", "policy_docs", "bol_samples", "save_all"]
try:
    missing = [f for f in expected if not hasattr(data, f)]
    car = data.carriers(20, seed=7)
    has_cols = set(car.columns) >= {"carrier_id", "carrier_name", "on_time_rate"}
    CHECK_ZORO = 1 if (not missing and has_cols) else 0
    print("zoro.data functions present:", not missing, "| missing:", missing)
    print("carriers(20, seed=7) ->", car.shape[0], "rows x", car.shape[1], "cols")
    print("OK: zoro.data generators callable." if CHECK_ZORO else "FAIL: zoro.data incomplete.")
except Exception as e:
    print("FAIL: zoro.data check raised:", e)

### Check 4：Gitがインストールされ、repo内にいるか

Gitは、**自分のwork**にとって、**data**に対するseedと同じreproducibility layerです。下の3つのcheck（Gitがある、work tree内にいる、branch上にいる）は、生成したdatasetをcommitするWeek 1金曜日のuse caseに進むための最低条件です。

In [ ]:
import subprocess

def run(args):
    return subprocess.run(args, capture_output=True, text=True)

CHECK_GIT = 0
try:
    gv = run(["git", "--version"])
    inside = run(["git", "rev-parse", "--is-inside-work-tree"]).stdout.strip() == "true"
    branch = run(["git", "branch", "--show-current"]).stdout.strip()
    print("git:", gv.stdout.strip())
    print("inside repo:", inside, "| branch:", branch)
    CHECK_GIT = 1 if (gv.returncode == 0 and inside) else 0
    print("OK: git present and inside a repository." if CHECK_GIT else "FAIL: git missing or not in a repo.")
except Exception as e:
    print("FAIL: git check raised:", e)

### NumPy refresher：arrayを思考の単位にする

このprogramのほぼすべてのmodelは、多次元arrayを通ります。実務の90%をカバーする4つの動作を確認します。arrayを作る、`shape`/`dtype`を調べる、axisに沿ってreduceする、そして **broadcast**（小さなarrayを対応するaxisで大きなarrayに合わせる）です。

In [ ]:
rng = np.random.default_rng(0)  # seed 0 for this demo only
a = rng.integers(0, 10, size=(3, 4))
print("array:")
print(a)
print("shape:", a.shape, "| dtype:", a.dtype)
print("column means:", a.mean(axis=0).round(3))

col_offsets = np.array([100, 200, 300, 400])
print("broadcast add (3,4) + (4,):")
print(a + col_offsets)

mask = a > 5
print("elements > 5:", int(mask.sum()), "of", a.size)

CHECK_NUMPY = 1
print("OK: numpy array/broadcast demo ran.")

### seedの習慣：programで最も大切な1行

AI outputは予測できません。これがprogram全体のdisciplineが対応する事実です。対抗策は、制御できる範囲での **determinism** です。*seed* は、pseudorandom generatorがどのrun・どのmachineでも同じsequenceを出すようにする固定integerです。modernで推奨される形は `numpy.random.default_rng(seed)` です。同じseedなら同じdataになるため、Week 1のdatasetをWeek 23でも再現できます。

In [ ]:
def first_five(seed):
    return np.random.default_rng(seed).integers(0, 1_000_000, size=5).tolist()

a = first_five(42)
b = first_five(42)
c = first_five(7)
print("seed 42 run 1:", a)
print("seed 42 run 2:", b)
print("seed  7 run  :", c)
print("same seed, same data:", a == b, "| different seed, different data:", a != c)
CHECK_SEED = 1 if a == b and a != c else 0
print("OK: determinism confirmed." if CHECK_SEED else "FAIL: determinism broken.")

### metricの習慣：self-checkも数字で終える

Week 3以降、すべてのAI artifactにはscoreを付けます。今から始めましょう。このnotebookのscoreは、6つのcheckのうち何個がpassしたかを表します。

In [ ]:
READINESS = CHECK_PYTHON + CHECK_LIBS + CHECK_ZORO + CHECK_GIT + CHECK_NUMPY + CHECK_SEED
print("READINESS_SCORE:", READINESS)
if READINESS == 6:
    print("Environment ready. Move to 02-zorologistics-data-generator.ipynb.")
else:
    print("Some checks failed - fix them before continuing.")